quantum-cortex — REPRISE step 4: the journal in the decode loop, fine tune from N1, the LM arm of the Molaison (H.M., after the patient Henry Molaison; defined in docs/benchmarks/hm-protocol.md) protocol (ADR-008)

Settings: Accelerator = GPU T4, Internet = On. Relay (optional, recommended): attach the secrets `KAGGLE_USERNAME` and `KAGGLE_KEY` (the values of your kaggle.json); the run then pushes its directory to a private Kaggle dataset after every checkpoint and a relaunch in a fresh container resumes from the last push instead of restarting. Inputs: the N1 notebook's version whose Output holds `runs/kaggle_t4/ckpt.pt` (the parent) and, if present, `data/fineweb_edu_bytes.bin` (the same slice; else it is rebuilt). Secret: `QUANTUM_CORTEX_JOURNAL_KEY`, 32 bytes hex, generated once locally with `python -c "from cortex_c2b.crypto import generate_key; print(generate_key().hex())"` and kept outside the repository: the run's journal is sealed with it, and a lost key is a lost journal.

What runs (set `CONFIG` in the checkpoint cell). v1 to v7 are RESULTS rows 21 to 29, each INVALID and attributed (`docs/RESULTS.md`): v6 (`configs/kaggle_t4_journal_v6.json`, run `81cb13a684aa`) is the reference recipe, the clean minimal pair with its shape kept. **v9** `configs/kaggle_t4_journal_v9.json` (the default below; ADR-008 amendment 2026-09-17, RES-23) is v6 plus one variable, the familiarity mark: every read window line carries the organ's own similarity for that line, `A(0.97):`, in training and at evaluation; the mark oracle (no model) is recorded in every hm-lm file as the declared ceiling. Measured 2026-09-18 (run `8ceba5db8d0b`, RESULTS row 36): the oracle passes, the model reads the mark in part. **v8** `configs/kaggle_t4_journal_v8.json` (declared the same day) is v6 plus `attn_local_conv` started as the identity, the reader with a local composition; not yet run. Thresholds untouched.

What it is: 120M tokens of fine tune from `be1fa8139f59`, half the steps on the organ use curriculum (the target of every example is computed from what the journal returned: cite what was shown, else abstain), the LM arm of the frozen protocol at the end (`metrics/mqar/hm-lm-<run_id>.json`, its fields in the record), then session B in a new process from the checkpoint and the sealed journal alone (`metrics/mqar/hm-lm-<run_id>-session-b.json`): the named boundary of ADR-007 D8, on the model. Wall time: about 2 h 10 of training on a T4 plus the arms, 2 h 15 in all (v6 2.10 GPU hours, v9 2.10). Two optional cells, each skipped unless its Input is mounted: the shape probe on the v5 checkpoint, and the mark oracle with the marks' distributions on the v6 checkpoint (RES-23).

Download the log as a file before reading any number. Then commit: `metrics/runs.jsonl` (the record line), `metrics/LATEST.md`, `metrics/mqar/hm-lm-<run_id>.json` (the in process arm) and `metrics/mqar/hm-lm-<run_id>-session-b.json` (the new process); the checkpoint and the sealed journal stay in the Output (where each run's artefacts live: `metrics/ARTEFACTS.md`).

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")   # the repository is private
!rm -rf quantum-cortex
!git clone https://{token}@github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
# the runs directory lives OUTSIDE the clone (/kaggle/working/runs-keep) and is never deleted: a session
# interruption that re-runs the notebook finds the last checkpoint there and resumes instead of restarting
import os
os.makedirs('/kaggle/working/runs-keep', exist_ok=True)
if not os.path.islink('runs'): os.symlink('/kaggle/working/runs-keep', 'runs')
print('runs ->', os.path.realpath('runs'), '| kept from a previous session:', os.listdir('runs'))
!pip -q install jsonschema datasets cryptography


In [ ]:
# the key: from the Kaggle secret to the environment, never to a file
import os
from kaggle_secrets import UserSecretsClient
os.environ['QUANTUM_CORTEX_JOURNAL_KEY'] = UserSecretsClient().get_secret('QUANTUM_CORTEX_JOURNAL_KEY')
assert len(bytes.fromhex(os.environ['QUANTUM_CORTEX_JOURNAL_KEY'])) == 32, 'the key must be 32 bytes hex'
print('journal key: present (not printed)')
# the checkpoint relay (cortex_data/relay.py): with these two secrets the run pushes its directory to a private
# Kaggle dataset after every checkpoint and pulls it back before a resume, so a relaunch in a fresh container
# restarts from the last thousand steps; without them everything runs as before
try:
    os.environ['KAGGLE_USERNAME'] = UserSecretsClient().get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = UserSecretsClient().get_secret('KAGGLE_KEY')
    from cortex_data import relay
    print('relay:', 'on' if relay.available() else 'off (kaggle command missing)')
except Exception as e:
    print('relay: off (secrets KAGGLE_USERNAME / KAGGLE_KEY not attached)')


In [ ]:
# the parent checkpoint (N1, runs/kaggle_t4/ckpt.pt) and the data slice, from the mounted Inputs
import os, shutil, glob, json
CONFIG = 'configs/kaggle_t4_journal_v9.json'      # the run to make: v9, the familiarity mark (RES-23); then ..._v8.json; earlier: ..._v7.json, ..._v6.json, ..._v5.json, ..._v4.json, ..._v3.json, ..._v2.json, kaggle_t4_journal.json
OUT = json.load(open(CONFIG))['out_dir']; print('config', CONFIG, '-> out_dir', OUT)
os.makedirs('runs/kaggle_t4', exist_ok=True); os.makedirs('data', exist_ok=True); os.makedirs(OUT, exist_ok=True)
import torch
PARENT_RUN_ID = json.load(open(CONFIG))['parent_run_id']
parent = None
for cand in sorted(glob.glob('/kaggle/input/**/ckpt.pt', recursive=True)):
    rid = torch.load(cand, map_location='cpu', weights_only=False).get('run_id')
    print('candidate', cand, 'run_id', rid)
    if rid == PARENT_RUN_ID: parent = cand
assert parent, f'no checkpoint with run_id {PARENT_RUN_ID} among the Inputs: mount the N1 version (its Output holds runs/kaggle_t4/ckpt.pt)'
shutil.copy(parent, 'runs/kaggle_t4/ckpt.pt'); print('parent checkpoint:', parent, 'run_id', PARENT_RUN_ID)
if os.path.exists(f'{OUT}/ckpt.pt'):
    print('own checkpoint kept in this session (resume after an interruption):', f'{OUT}/ckpt.pt')
else:
    own = sorted(glob.glob(f'/kaggle/input/**/{os.path.basename(OUT)}/ckpt.pt', recursive=True))
    if own: shutil.copy(own[-1], f'{OUT}/ckpt.pt'); print('own checkpoint recovered from an Input (resume; train.py refuses it if the config hash differs):', own[-1])
data = sorted(glob.glob('/kaggle/input/**/data/fineweb_edu_bytes.bin', recursive=True))
if data:
    shutil.copy(data[0], 'data/fineweb_edu_bytes.bin'); print('data slice reused:', data[0])
else:
    import numpy as np
    from datasets import load_dataset
    budget = 600_000_000
    out = np.memmap('data/fineweb_edu_bytes.bin', dtype=np.uint16, mode='w+', shape=(budget,))
    ds = load_dataset('HuggingFaceFW/fineweb-edu', name='sample-10BT', split='train', streaming=True)
    i = 0
    for ex in ds:
        b = ex['text'].encode('utf-8', errors='ignore'); m = min(len(b), budget - i)
        out[i:i+m] = np.frombuffer(b[:m], dtype=np.uint8); i += m
        if i >= budget: break
    out.flush(); print(f'data slice rebuilt: {i:,} byte-tokens')

# the relay's last push, if any, before the resume lookup below is trusted
from cortex_data import relay
SLUG = relay.slug_for(os.path.basename(OUT))
if relay.available() and not os.path.exists(f'{OUT}/ckpt.pt'):
    relay.pull(SLUG, OUT)
print('relay slug:', SLUG, '| resume checkpoint present:', os.path.exists(f'{OUT}/ckpt.pt'))


In [ ]:
# OPTIONAL: the shape probe on the v5 checkpoint (run db028e6a4262). Skipped unless the v5 notebook version is mounted as an
# Input (Add Input -> the version whose Output holds runs-keep/journal-n1-v5). A diagnostic outside the frozen protocol, like
# the memorisation probe: the same planted fact with its episode withheld, at one line fewer (the v5 negative as trained) and
# at the read's own number of lines (the v6 negative); the never planted entities at the read's number of lines (the protocol)
# and at one fewer. About a minute of T4. Nothing here touches the v6 run.
import glob, os, shutil
V5_CONFIG = 'configs/kaggle_t4_journal_v5.json'
cands = sorted(glob.glob('/kaggle/input/**/journal-n1-v5/ckpt.pt', recursive=True))
src = next((os.path.dirname(c) for c in cands if os.path.exists(os.path.join(os.path.dirname(c), 'journal.jsonl'))), None)
if src is None:
    print('shape probe skipped: no Input holds journal-n1-v5/ckpt.pt next to its journal.jsonl (mount the v5 version to run it)')
else:
    dst = 'runs/journal-n1-v5-probe'                    # a writable copy: the checkpoint, the sealed journal, its payload directory
    if os.path.exists(dst): shutil.rmtree(dst)
    shutil.copytree(src, dst); print('shape probe on a copy of', src)
    !python -m cortex_c2b.hm_lm --shape-probe --ckpt {dst}/ckpt.pt --journal {dst}/journal.jsonl --config {V5_CONFIG}
    shutil.rmtree(dst, ignore_errors=True)               # the copy is not an artefact; the probe's json under metrics/mqar/ is


In [ ]:
# OPTIONAL: the mark oracle and the marks' distributions on the v6 checkpoint (RES-23, ADR-008 amendment 2026-09-17): the ceiling
# the organ's similarities already allowed under v6's own cue encoder, read BEFORE v9 runs, at no GPU cost. Skipped unless the v6
# notebook version is mounted as an Input (Add Input -> the version whose Output holds runs-keep/journal-n1-v6). Session B in a new
# process on v6's checkpoint and sealed journal; the file carries hm_marks and hm_mark_oracle next to the same readouts as row 24.
import glob, os, shutil
V6_CONFIG = 'configs/kaggle_t4_journal_v6.json'
cands = sorted(glob.glob('/kaggle/input/**/journal-n1-v6/ckpt.pt', recursive=True))
if cands:
    v6dir = os.path.dirname(cands[0])
    os.makedirs('runs-keep/journal-n1-v6', exist_ok=True)
    for f in os.listdir(v6dir):
        s, d = os.path.join(v6dir, f), os.path.join('runs-keep/journal-n1-v6', f)
        (shutil.copytree if os.path.isdir(s) else shutil.copy)(s, d) if not os.path.exists(d) else None
    !python -m cortex_c2b.hm_lm --session-b --ckpt runs-keep/journal-n1-v6/ckpt.pt --journal runs-keep/journal-n1-v6/journal.jsonl --config {V6_CONFIG} --out metrics/mqar/hm-lm-81cb13a684aa-session-b-marks.json
else:
    print('v6 checkpoint not mounted: the mark oracle on v6 is skipped (attach the v6 version as Input to run it)')


In [ ]:
flag = '--resume' if os.path.exists(f'{OUT}/ckpt.pt') else ''
!python train.py --config {CONFIG} {flag} --time-budget-min 645 --relay {SLUG}


In [ ]:
# the named boundary, on the model: session B in a NEW process, from the checkpoint and the sealed journal alone;
# plus the memorisation probe (the contract on the run's first training pool, entities seen in training)
!python -m cortex_c2b.hm_lm --session-b --ckpt {OUT}/ckpt.pt --journal {OUT}/journal.jsonl --config {CONFIG} --probe-training-pool


In [ ]:
import json
a = json.load(open(f'{OUT}/hm-lm.json')); import glob, os; b = json.load(open(max(glob.glob('metrics/mqar/hm-lm-*-session-b.json'), key=os.path.getmtime)))   # the newest, never the alphabetically last
print('in process :', a['verdict']); print('  ', {k: a[k] for k in ('hm_recall_on','hm_recall_off','hm_skill_delta','hm_invalid_citation_on','hm_false_abstention_on','hm_negctrl_rate','persistent','claimable')})
print('  attribution (encoder, reader, answer):', a['attribution']); print('  match head (present on planted, absent on never planted):', a.get('hm_match_acc_on'), a.get('hm_match_acc_negctrl'))
print('new process:', {k: b[k] for k in ('hm_recall_on','hm_recall_off','hm_invalid_citation_on','hm_false_abstention_on','hm_retrieval_hit','persistent')})
same = all(a[k] == b[k] for k in ('hm_recall_on','hm_recall_off','hm_invalid_citation_on','hm_false_abstention_on','hm_retrieval_hit'))
print('session B reproduces the in process arm:', same)
print('training pool probe (seen entities) :', b.get('training_pool_probe'))
print(open('metrics/LATEST.md').read()); print(open('metrics/runs.jsonl').read().splitlines()[-1])
print('Commit: metrics/runs.jsonl, metrics/LATEST.md, metrics/mqar/hm-lm-<run_id>.json, metrics/mqar/hm-lm-<run_id>-session-b.json. Read cold first.')


In [ ]:
# a manifest of the artefacts with their hashes, to verify any later retrieval
import hashlib, glob, json, os
files = sorted(glob.glob('metrics/mqar/hm-lm-*.json') + glob.glob('metrics/mqar/shape-probe-*.json') + glob.glob(f'{OUT}/*') + glob.glob(f'{OUT}/journal.payloads/*'))
manifest = {f: {'bytes': os.path.getsize(f), 'sha256_16': hashlib.sha256(open(f, 'rb').read()).hexdigest()[:16]} for f in files if os.path.isfile(f)}
open(f'{OUT}/MANIFEST.json', 'w').write(json.dumps(manifest, indent=2))
print(json.dumps({k: v for k, v in manifest.items() if 'payloads' not in k}, indent=1)); print(len(manifest), 'files in the manifest')
